# Randomization wrapper - correctness checks

Run this notebook from the `part2/` folder so that `rand_wrapper` is importable.

In [ ]:
import gymnasium as gym
import panda_gym  # noqa: F401 - registers the Panda environments
from rand_wrapper import RandomizationWrapper


def make(env_type="source", strategy="none", mass_range=(0.5, 8.0), seed=0):
    e = gym.make("PandaPush-v3", render_mode="rgb_array", type=env_type, reward_type="dense")
    return RandomizationWrapper(e, mass_range=mass_range, mode=strategy, seed=seed)


## UDR samples within the given range

In [ ]:
w = make(strategy="udr", mass_range=(0.5, 8.0), seed=1)
masses = [w._sample_mass() for _ in range(500)]
assert all(0.5 <= m <= 8.0 for m in masses)
print("ok: 500 UDR samples all in [0.5, 8.0], e.g.", round(masses[0], 3))


## `none` does not change the mass

In [ ]:
assert make(strategy="none")._sample_mass() is None
print("ok: none leaves the object mass untouched")


## ADR boundaries stay inside the global limits

In [ ]:
w = make(strategy="adr", mass_range=(0.5, 8.0), seed=2)
assert 0.5 <= w.phi_low <= w.phi_high <= 8.0
print("ok: phi =", (round(w.phi_low, 3), round(w.phi_high, 3)))


## The same seed is reproducible

In [ ]:
a = make(strategy="udr", seed=7)
b = make(strategy="udr", seed=7)
assert [a._sample_mass() for _ in range(20)] == [b._sample_mass() for _ in range(20)]
print("ok: identical mass sequence for seed 7")


## The sampled mass is actually applied to the simulator

In [ ]:
w = make(strategy="udr", mass_range=(5.0, 5.0), seed=0)  # degenerate range -> mass == 5.0
w.reset()
sim = w.env.unwrapped.task.sim
mass = sim.physics_client.getDynamicsInfo(sim._bodies_idx["object"], -1)[0]
assert abs(mass - 5.0) < 1e-6
print("ok: object mass read back from PyBullet =", mass)


## Source and target environments differ

In [ ]:
def obj_mass(env_type):
    w = make(env_type=env_type)
    w.reset()
    s = w.env.unwrapped.task.sim
    return s.physics_client.getDynamicsInfo(s._bodies_idx["object"], -1)[0]

ms, mt = obj_mass("source"), obj_mass("target")
assert ms != mt
print("ok: source mass", ms, "!= target mass", mt)
